In [ ]:
import fitz  # PyMuPDF库，用于处理PDF文件
import os  # 操作系统相关功能
import numpy as np  # NumPy库，用于数值计算
import json  # JSON数据处理
from openai import OpenAI  # OpenAI API客户端

In [ ]:
def get_embedding(text, model="text-embedding-ada-002"):
    """
    使用指定模型为给定文本生成嵌入向量。

    参数:
    text (str): 输入文本。
    model (str): 嵌入模型名称。

    返回:
    np.ndarray: 嵌入向量。
    """
    response = client.embeddings.create(model=model, input=text)
    return np.array(response.data[0].embedding)


In [ ]:

# 将文本按句子分割（基本分割）
sentences = extracted_text.split(". ")

# 为每个句子生成嵌入向量
embeddings = [get_embedding(sentence) for sentence in sentences]


In [ ]:
def cosine_similarity(vec1, vec2):
    """
    计算两个向量之间的余弦相似度。

    参数:
    vec1 (np.ndarray): 第一个向量。
    vec2 (np.ndarray): 第二个向量。

    返回:
    float: 余弦相似度。
    """
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# 计算连续句子之间的相似度
similarities = [cosine_similarity(embeddings[i], embeddings[i + 1]) for i in range(len(embeddings) - 1)]

In [ ]:
def compute_breakpoints(similarities, method="percentile", threshold=90):
    """
    根据相似度下降计算分块断点。

    参数:
    similarities (List[float]): 句子之间的相似度分数列表。
    method (str): 'percentile', 'standard_deviation' 或 'interquartile'。
    threshold (float): 阈值（对于 'percentile' 是百分位数，对于 'standard_deviation' 是标准差的数量）。

    返回:
    List[int]: 应该发生分块分裂的索引位置列表。
    """
    # 根据所选方法确定阈值
    if method == "percentile":
        # 计算相似度分数的第X百分位
        threshold_value = np.percentile(similarities, threshold)
    elif method == "standard_deviation":
        # 计算相似度分数的平均值和标准差
        mean = np.mean(similarities)
        std_dev = np.std(similarities)
        # 将阈值设置为平均值减去X个标准差
        threshold_value = mean - (threshold * std_dev)
    elif method == "interquartile":
        # 计算第一和第三四分位数（Q1 和 Q3）
        q1, q3 = np.percentile(similarities, [25, 75])
        # 使用IQR规则设置阈值以检测异常值
        threshold_value = q1 - 1.5 * (q3 - q1)
    else:
        # 如果提供了无效方法，则引发错误
        raise ValueError("Invalid method. Choose 'percentile', 'standard_deviation', or 'interquartile'.")

    # 找出相似度低于阈值的位置索引
    return [i for i, sim in enumerate(similarities) if sim < threshold_value]

# 使用百分位法并设置阈值为90计算断点
breakpoints = compute_breakpoints(similarities, method="percentile", threshold=90)

In [ ]:
def split_into_chunks(sentences, breakpoints):
    """
    将句子分割为语义块。

    参数:
    sentences (List[str]): 句子列表。
    breakpoints (List[int]): 应该发生分割的索引位置。

    返回:
    List[str]: 文本块列表。
    """
    chunks = []  # 初始化一个空列表来存储块
    start = 0  # 初始化起始索引

    # 遍历每个断点以创建块
    for bp in breakpoints:
        # 将从起始到当前断点的句子片段加入块中
        chunks.append(". ".join(sentences[start:bp + 1]) + ".")
        start = bp + 1  # 更新起始索引到断点后的下一个句子

    # 将剩余的句子作为最后一个块加入
    chunks.append(". ".join(sentences[start:]))
    return chunks  # 返回块列表

# 使用 split_into_chunks 函数创建块
text_chunks = split_into_chunks(sentences, breakpoints)
